In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_sub = pd.read_csv("sample_submission.csv")

# 2. Separate features (X) and target (y) from train.csv
X = train.drop(columns=["id", "is_fraud"])
y = train["is_fraud"]
X_test = test.drop(columns=["id"])

# 3. One-hot encode categorical columns together so train and test have identical columns
cat_cols = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
all_features = pd.concat([X, X_test], axis=0)
all_encoded = pd.get_dummies(all_features, columns=cat_cols, drop_first=True)

# Split back into X (train) and X_test (test)
X = all_encoded.iloc[:len(train)].copy()
X_test = all_encoded.iloc[len(train):].copy()

# 4. Create a validation split from train.csv to test your model's accuracy
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Train the Random Forest model (balanced for rare fraud cases)
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# 6. Check validation performance
val_probs = model.predict_proba(X_val)[:, 1]
print("Validation ROC-AUC:", round(roc_auc_score(y_val, val_probs), 4))
print("Validation PR-AUC:", round(average_precision_score(y_val, val_probs), 4))

# 7. Predict on test.csv and match the exact ID order of sample_submission.csv
test_preds = pd.DataFrame({
    "id": test["id"],
    "is_fraud": model.predict_proba(X_test)[:, 1]
})

# Merge onto sample_submission's 'id' column to guarantee exact row order
final_submission = sample_sub[["id"]].merge(test_preds, on="id", how="left")
final_submission.to_csv("my_submission.csv", index=False)

print("Saved my_submission.csv successfully! Rows:", len(final_submission))

Validation ROC-AUC: 0.9305
Validation PR-AUC: 0.3721
Saved my_submission.csv successfully! Rows: 299984


In [14]:
import pandas as pd

# Load your test data and predictions
danger_df = pd.read_csv("my_submission.csv").rename(columns={"is_fraud": "fraud_probability"})
test = pd.read_csv("test.csv")

# Merge full behavioral details back onto your flagged high-risk list
scam_check = danger_df.merge(test, on="id", how="inner")

# Filter specifically for customers showing active SCAM victim signs:
# 1. AI scam attempt flagged, OR
# 2. High fraud probability + sending money to a new merchant + draining >70% of balance
scam_victims = scam_check[
    (scam_check["is_ai_generated_scam_attempt"] == True) |
    (
        (scam_check["fraud_probability"] >= 0.70) &
        (scam_check["is_new_merchant"] == True) &
        (scam_check["amount_usd"] > scam_check["account_balance_usd"] * 0.70)
    )
].sort_values(by="fraud_probability", ascending=False).copy()

# Summarize which active scam signals applied to each customer.
scam_victims["scam_indicators"] = scam_victims.apply(
    lambda row: " | ".join([
        name for name, active in {
            "AI-generated scam attempt": row["is_ai_generated_scam_attempt"],
            "High fraud probability": row["fraud_probability"] >= 0.70,
            "New merchant": row["is_new_merchant"],
            "Transfer exceeds 70% of balance": row["amount_usd"] > row["account_balance_usd"] * 0.70,
        }.items() if active
    ]), axis=1
)

# Print the exact customer IDs in danger of getting scammed
print(f"Total customers in active danger of getting scammed: {len(scam_victims)}")
print("\n--- CUSTOMERS IN DANGER OF GETTING SCAMMED ---")
print(scam_victims[[
    "id", "fraud_probability", "amount_usd", "account_balance_usd",
    "is_ai_generated_scam_attempt", "scam_indicators"
]].to_string(index=False))

# Save just the scam victims to their own priority call list
scam_victims[["id", "fraud_probability", "amount_usd", "scam_indicators"]].to_csv(
    "active_scam_victims.csv", index=False
)


Total customers in active danger of getting scammed: 1187

--- CUSTOMERS IN DANGER OF GETTING SCAMMED ---
    id  fraud_probability  amount_usd  account_balance_usd  is_ai_generated_scam_attempt                                                            scam_indicators
972810           0.976647     181.480               766.07                          True          AI-generated scam attempt | High fraud probability | New merchant
759904           0.960025      38.210              1649.83                          True          AI-generated scam attempt | High fraud probability | New merchant
904089           0.959195      27.600              1087.65                          True          AI-generated scam attempt | High fraud probability | New merchant
883645           0.951626     562.580               489.24                         False    High fraud probability | New merchant | Transfer exceeds 70% of balance
936544           0.947953      40.410               618.59                